<a href="https://colab.research.google.com/github/LouisFOU/Solar-Panel-Detection/blob/main/SolarSegmentationPipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# installing

!pip install "numpy<2.0.0" "torchgeo>=0.6.0" terratorch h5py lightning

print("Done, restart kernel")

In [ ]:
#import libraries
import numpy, torch, lightning, h5py, torchgeo
from google.colab import drive

print("NumPy version :", numpy.__version__)

print("Connecting Google Drive...")
drive.mount('/content/drive')


In [ ]:
# Imports
from typing import Dict, Tuple
import os
import glob
import h5py
import numpy as np
from pathlib import Path

import torch
from torch.utils.data import Dataset, DataLoader

import lightning.pytorch as pl
from lightning.pytorch import Trainer
from lightning.pytorch.callbacks import ModelCheckpoint

# Foundation Models import
import terratorch
import terratorch.models

from terratorch.tasks import SemanticSegmentationTask

from google.colab import drive
drive.mount('/content/drive')

#Files on drive
DATA_DIR = Path("/content/dataset_local")
OUTPUT_DIR = Path("/content/drive/MyDrive/pv_ai_outputs")
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Hyperparameters
BATCH_SIZE   = 2      # Réduire à 2 si erreur de mémoire "OOM" (Out Of Memory)
NUM_EPOCHS   = 5      # 5 pour tester, à monter à 20+ plus tard
LEARNING_RATE = 1e-4
NUM_WORKERS  = 0      # Sur Colab, 2 fonctionne généralement très bien
IMG_SIZE     = 224    # Taille d'entrée de Prithvi
NUM_CLASSES  = 2      # 0=fond, 1=panneau solaire

PRITHVI_BANDS = ["BLUE", "GREEN", "RED", "NIR_NARROW", "SWIR_1", "SWIR_2"]

print(f"   Data read from : {DATA_DIR}")
print(f"   Models saved in : {OUTPUT_DIR}")
print(f"   Bands used    : {PRITHVI_BANDS}")
print(f"Checkpoints redirected vers : {CHECKPOINT_DIR}")

In [ ]:

import h5py
import numpy as np
from pathlib import Path

def analyse_pv4ger_structure(data_dir: Path):
    # Looking for files
    hdf5_files = list(data_dir.glob("*.hdf5"))
    if not hdf5_files:
        hdf5_files = list(data_dir.rglob("*.hdf5"))

    print(f"{len(hdf5_files)} files .hdf5 found in {data_dir}")

    # Opens the first one
    sample_file = hdf5_files[0]
    with h5py.File(sample_file, 'r') as f:
        print(f"Keys found : {list(f.keys())}")

        blue_data = f['Blue'][...]
        blue_shape = blue_data.shape

        # Mask analysis
        label_data = f['label'][...]
        label_shape = label_data.shape
        label_unique = np.unique(label_data)

        print(f"\n Format des images détecté : 3 bandes séparées")
        print(f"   Dimensions par bande : {blue_shape} | Type : {blue_data.dtype}")
        print(f"   Valeurs des pixels : Min = {blue_data.min()} | Max = {blue_data.max()}")

        print(f"\n Masque (label) détecté :")
        print(f"   Dimensions : {label_shape} | Type : {label_data.dtype}")
        print(f"   Classes présentes : {label_unique} (0 = fond, 1 = panneau)")

    return hdf5_files

ALL_FILES = analyse_pv4ger_structure(DATA_DIR)

N_BANDS = 3
PRITHVI_BANDS = ["BLUE", "GREEN", "RED"]

print(f" Updated config : Use of {N_BANDS} bands : {PRITHVI_BANDS}")

In [ ]:
# ==============================================================
# CELLULE 4 : DATASET ET DATAMODULE (FUSION CLAUDE + CORRECTION RGB)
# ==============================================================

import json
import random
import h5py
import numpy as np
from pathlib import Path
from typing import Optional, List, Dict, Tuple

import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
import lightning.pytorch as pl

# ── 1. RAPPEL DES CHEMINS (Pour trouver les JSON sur le Drive) ──
DRIVE_DIR = Path("/content/drive/MyDrive/m-pv4ger-seg")
PARTITION_PATH = DRIVE_DIR / "partition.json"
BAND_STATS_PATH = DRIVE_DIR / "band_stats.json"

class PV4GERDataset(Dataset):
    def __init__(self, file_paths: List[Path], img_size: int = 224, augment: bool = False):
        super().__init__()
        self.file_paths = file_paths
        self.img_size   = img_size
        self.augment    = augment

    def __len__(self) -> int:
        return len(self.file_paths)

    def _read_and_stack_hdf5(self, path: Path) -> Tuple[np.ndarray, np.ndarray]:
        """Lecture sur mesure pour les 3 bandes séparées de ton dataset PV4GER."""
        with h5py.File(path, 'r') as f:
            # On lit et on empile directement dans l'ordre de PRITHVI_BANDS
            blue = f['Blue'][...]
            green = f['Green'][...]
            red = f['Red'][...]
            # Résultat : un tableau de forme (3, 320, 320)
            image = np.stack([blue, green, red], axis=0).astype(np.float32)

            label = f['label'][...].astype(np.int64)
        return image, label

    def _augment(self, image: torch.Tensor, label: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """Augmentations spatiales pour éviter le surapprentissage."""
        if random.random() > 0.5:
            image = TF.hflip(image)
            label = TF.hflip(label.unsqueeze(0)).squeeze(0)
        if random.random() > 0.5:
            image = TF.vflip(image)
            label = TF.vflip(label.unsqueeze(0)).squeeze(0)

        angle = random.choice([0, 90, 180, 270])
        if angle > 0:
            image = TF.rotate(image, angle)
            label = TF.rotate(label.unsqueeze(0), angle).squeeze(0)
        return image, label

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        path = self.file_paths[idx]

        # 1. Lecture et empilage des couleurs
        image, label = self._read_and_stack_hdf5(path)

        # 2. Normalisation très stable (diviser par 255 car on sait que c'est du uint8)
        image = image / 255.0

        # 3. Conversion en Tenseurs PyTorch
        image = torch.from_numpy(image)       # (3, H, W) float32
        label = torch.from_numpy(label)       # (H, W)    int64

        # 4. Redimensionnement (Prithvi veut du 224x224)
        if image.shape[-1] != self.img_size or image.shape[-2] != self.img_size:
            image = TF.resize(image, [self.img_size, self.img_size], interpolation=TF.InterpolationMode.BILINEAR, antialias=True)
            # Attention : on utilise NEAREST pour le masque pour ne pas créer de classes "0.5" !
            label = TF.resize(label.unsqueeze(0), [self.img_size, self.img_size], interpolation=TF.InterpolationMode.NEAREST).squeeze(0)

        # 5. Augmentations (mode train uniquement)
        if self.augment:
            image, label = self._augment(image, label)

        return {"image": image, "mask": label}


class PV4GERDataModule(pl.LightningDataModule):
    def __init__(self, data_dir: Path, partition_path: Path, img_size: int = 224, batch_size: int = 4, num_workers: int = 2):
        super().__init__()
        self.data_dir       = data_dir
        self.partition_path = partition_path
        self.img_size       = img_size
        self.batch_size     = batch_size
        self.num_workers    = num_workers
        self.all_files      = list(data_dir.glob("*.hdf5"))

    def _load_partition(self) -> Tuple[List[Path], List[Path], List[Path]]:
        file_map = {f.name: f for f in self.all_files}

        if self.partition_path.exists():
            with open(self.partition_path) as f:
                partition = json.load(f)
            train_files = [file_map[n] for n in partition.get("train", []) if n in file_map]
            val_files   = [file_map[n] for n in partition.get("val",   []) if n in file_map]
            test_files  = [file_map[n] for n in partition.get("test",  []) if n in file_map]
            print(f"✅ Partition JSON chargée : train={len(train_files)}, val={len(val_files)}, test={len(test_files)}")
            return train_files, val_files, test_files

        print("⚠️ partition.json introuvable sur le Drive → split automatique 80/10/10")
        np.random.seed(42)
        indices = np.random.permutation(len(self.all_files))
        n = len(self.all_files)
        return [self.all_files[i] for i in indices[:int(0.8*n)]], \
               [self.all_files[i] for i in indices[int(0.8*n):int(0.9*n)]], \
               [self.all_files[i] for i in indices[int(0.9*n):]]

    def setup(self, stage: Optional[str] = None):
        train_files, val_files, test_files = self._load_partition()

        self.train_dataset = PV4GERDataset(train_files, self.img_size, augment=True)
        self.val_dataset   = PV4GERDataset(val_files, self.img_size, augment=False)
        self.test_dataset  = PV4GERDataset(test_files, self.img_size, augment=False)

    def train_dataloader(self) -> DataLoader:
        return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True, num_workers=self.num_workers, pin_memory=torch.cuda.is_available(), drop_last=True)

    def val_dataloader(self) -> DataLoader:
        return DataLoader(self.val_dataset, batch_size=self.batch_size, shuffle=False, num_workers=self.num_workers, pin_memory=torch.cuda.is_available())

    def test_dataloader(self) -> DataLoader:
        return DataLoader(self.test_dataset, batch_size=self.batch_size, shuffle=False, num_workers=self.num_workers, pin_memory=torch.cuda.is_available())

# ── Instanciation et Test ──
print("⏳ Préparation du flux de données...")
datamodule = PV4GERDataModule(data_dir=DATA_DIR, partition_path=PARTITION_PATH, img_size=IMG_SIZE, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)
datamodule.setup()

sample_batch = next(iter(datamodule.train_dataloader()))
print(f"\n🔍 Vérification finale du batch pour Prithvi :")
print(f"   Shape Image : {sample_batch['image'].shape}  (Attendu: B, 3, 224, 224)")
print(f"   Shape Masque: {sample_batch['mask'].shape}  (Attendu: B, 224, 224)")
print(f"   Valeurs Min/Max Image: [{sample_batch['image'].min():.3f}, {sample_batch['image'].max():.3f}]")

In [ ]:
# ==============================================================
# CELLULE 5 : INITIALISATION DE PRITHVI ET ENTRAÎNEMENT (FINALE & ANTI-PARESSE)
# ==============================================================

import torch
import lightning.pytorch as pl
from lightning.pytorch.loggers import TensorBoardLogger, CSVLogger
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping, LearningRateMonitor
from terratorch.tasks import SemanticSegmentationTask

print("🧠 Chargement du modèle Prithvi EO v2 (300M) depuis IBM/NASA...")

# ── 1. CRÉATION DU MODÈLE (AVEC ARME ANTI-PARESSE) ──
model = SemanticSegmentationTask(
    model_args={
        "backbone": "prithvi_eo_v2_300",
        "decoder": "FCNDecoder",
        "backbone_pretrained": True,
        "backbone_bands": PRITHVI_BANDS,
        "num_classes": NUM_CLASSES,
    },
    model_factory="EncoderDecoderFactory",
    loss="ce",
    class_weights=[1.0, 15.0],   # 🚨 CORRECTIF MAJEUR : Rater un panneau (classe 1) coûte 15x plus cher !
    optimizer="AdamW",
    lr=5e-5,                     # 🚨 On baisse un peu la vitesse pour un apprentissage plus fin
    plot_on_val=False,
    ignore_index=-1
)

# ── 2. LOGGERS ──
tb_logger  = TensorBoardLogger(save_dir=str(OUTPUT_DIR), name="prithvi_pv4ger")
csv_logger = CSVLogger(save_dir=str(OUTPUT_DIR), name="prithvi_pv4ger")

# ── 3. CALLBACKS ──
callbacks_list = [
    # Sauvegarde basée sur la précision du détourage (mIoU)
    ModelCheckpoint(
        dirpath    = str(CHECKPOINT_DIR),
        filename   = "prithvi-pv4ger-{epoch:02d}-{val_mIoU:.4f}",
        monitor    = "val/mIoU",
        mode       = "max",                  # On veut que la précision (mIoU) soit MAXIMALE
        save_top_k = 1,
        verbose    = True,
    ),
    # Early Stopping basé sur la perte de validation
    EarlyStopping(
        monitor  = "val/loss",
        patience = 5,
        mode     = "min",                    # On veut que l'erreur globale (loss) soit MINIMALE
        verbose  = True,
    ),
    LearningRateMonitor(logging_interval="epoch"),
]

# ── 4. TRAINER ──
trainer = pl.Trainer(
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    devices=1,
    precision="16-mixed",
    max_epochs=NUM_EPOCHS,
    logger=[tb_logger, csv_logger],
    log_every_n_steps=5,
    callbacks=callbacks_list,
    gradient_clip_val=1.0,
    deterministic=False,
    num_sanity_val_steps=2,
)

print("\n" + "="*60)
print("🚀 Lancement de l'entraînement avec Poids de Classes... C'est parti !")
print("="*60 + "\n")

# ── 🔥 LANCEMENT ──
trainer.fit(
    model      = model,
    datamodule = datamodule,
)

print("\n" + "="*60)
print("✅ ENTRAÎNEMENT TERMINÉ !")
print(f"   Meilleur checkpoint sauvegardé ici : {trainer.checkpoint_callback.best_model_path}")

In [ ]:
# ==============================================================
# CELLULE 6 : CHARGEMENT DU CHAMPION ET VISUALISATION HEATMAP
# ==============================================================

import torch
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from terratorch.tasks import SemanticSegmentationTask

device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. Le chemin exact de ton meilleur modèle (vu dans tes logs précédents) !
best_checkpoint = "/content/drive/MyDrive/pv_ai_outputs/checkpoints/prithvi-pv4ger-epoch=02-val_mIoU=0.0000.ckpt"

print("🧠 Chargement exclusif du Champion (Époque 2) depuis le Drive...")
champion_model = SemanticSegmentationTask.load_from_checkpoint(best_checkpoint)
champion_model.eval()
champion_model = champion_model.to(device)

print("✅ Modèle chargé avec succès ! Préparation des images...")

# 2. La fonction de visualisation
def visualize_random_heatmap(model, dataset, n_samples=4, device="cuda"):
    indices = random.sample(range(len(dataset)), n_samples)

    images = torch.stack([dataset[i]["image"] for i in indices]).to(device)
    masks  = torch.stack([dataset[i]["mask"] for i in indices]).to(device)

    with torch.no_grad():
        output = model(images)
        logits = output.output if hasattr(output, 'output') else output
        # Probabilité de la classe 1 (Panneaux solaires)
        probabilities = torch.softmax(logits, dim=1)[:, 1, :, :].cpu()

    images = images.cpu()
    masks  = masks.cpu()

    fig, axes = plt.subplots(n_samples, 3, figsize=(12, 4 * n_samples))
    if n_samples == 1: axes = np.expand_dims(axes, axis=0)

    cmap_seg = plt.cm.colors.ListedColormap(['#4a4e69', '#f4a261'])

    for i in range(n_samples):
        img_rgb = images[i][[2, 1, 0]].permute(1, 2, 0).numpy()
        img_rgb = np.clip(img_rgb, 0, 1)

        axes[i, 0].imshow(img_rgb)
        axes[i, 0].set_title(f"Image Originale", fontsize=11)
        axes[i, 0].axis("off")

        axes[i, 1].imshow(masks[i].numpy(), cmap=cmap_seg, vmin=0, vmax=1)
        axes[i, 1].set_title("Vérité Terrain", fontsize=11)
        axes[i, 1].axis("off")

        im = axes[i, 2].imshow(probabilities[i].numpy(), cmap='magma', vmin=0, vmax=1)
        axes[i, 2].set_title("Confiance IA (Heatmap)", fontsize=11)
        axes[i, 2].axis("off")

    cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
    cbar = fig.colorbar(im, cax=cbar_ax)
    cbar.set_label("Probabilité (0.0 à 1.0)", rotation=270, labelpad=15)

    plt.suptitle("Prithvi EO V2 - Test du Modèle avec Poids de Classes", fontsize=14, fontweight="bold", y=0.95)
    plt.savefig(OUTPUT_DIR / "visualisation_champion.png", dpi=150, bbox_inches="tight")
    plt.show()

# 3. On lance le dessin !
visualize_random_heatmap(champion_model, datamodule.val_dataset, n_samples=4, device=device)